In [13]:
import requests
import pandas as pd
import numpy as np
import polars as pl

In [14]:
# Fetch help contacts data
help_contacts = requests.get("http://api.justgood.work/api/v2/help-contacts")
help_contacts_json = help_contacts.json()["data"]
help_contacts_json

[{'_id': '648ddd6fd09d794ae0d96902',
  'deviceId': 'jgw-vanilla-91aa897b5dd46640-TJK -> GBR',
  'appId': 'jgw-vanilla',
  'locale': 'TJK',
  'route': 'TJK -> GBR',
  'phoneNumber': '0300 400 100',
  'organization': 'Seasonal Worker Scheme - Pro-Force',
  'email': '',
  'createdAt': '2023-06-17T16:21:03.249Z',
  'updatedAt': '2023-06-17T16:21:03.249Z',
  '__v': 0},
 {'_id': '6483766dbf8be67e27fb4255',
  'deviceId': 'jgw-vanilla-c5628e281204e8a8-BGD -> GBR',
  'appId': 'jgw-vanilla',
  'locale': 'BGD',
  'route': 'BGD -> GBR',
  'phoneNumber': '08000 121 700',
  'organization': '',
  'email': '',
  'createdAt': '2023-06-09T18:58:53.695Z',
  'updatedAt': '2023-06-09T18:58:53.695Z',
  '__v': 0},
 {'_id': '648103f3bf8be67e27fb4200',
  'deviceId': 'jgw-vanilla-1a4be771d1a85da5-MYS -> MYS',
  'appId': 'jgw-vanilla',
  'locale': 'MYS',
  'route': 'MYS -> MYS',
  'phoneNumber': ' +60195122852',
  'organization': '',
  'email': '',
  'createdAt': '2023-06-07T22:25:55.142Z',
  'updatedAt': '2023-

In [15]:
# Convert help contacts json to a pandas Dataframe
hc_df = pd.DataFrame.from_records(help_contacts_json)

# view the first 2 records
hc_df.head()

,_id,deviceId,appId,locale,route,phoneNumber,organization,email,createdAt,updatedAt,__v
0,648ddd6fd09d794ae0d96902,jgw-vanilla-91aa897b5dd46640-TJK -> GBR,jgw-vanilla,TJK,TJK -> GBR,0300 400 100,Seasonal Worker Scheme - Pro-Force,,2023-06-17T16:21:03.249Z,2023-06-17T16:21:03.249Z,0
1,6483766dbf8be67e27fb4255,jgw-vanilla-c5628e281204e8a8-BGD -> GBR,jgw-vanilla,BGD,BGD -> GBR,08000 121 700,,,2023-06-09T18:58:53.695Z,2023-06-09T18:58:53.695Z,0
2,648103f3bf8be67e27fb4200,jgw-vanilla-1a4be771d1a85da5-MYS -> MYS,jgw-vanilla,MYS,MYS -> MYS,+60195122852,,,2023-06-07T22:25:55.142Z,2023-06-07T22:25:55.142Z,0
3,648103ecbf8be67e27fb41ff,jgw-vanilla-1a4be771d1a85da5-MYS -> MYS,jgw-vanilla,MYS,MYS -> MYS,+60195122852,,,2023-06-07T22:25:48.413Z,2023-06-07T22:25:48.413Z,0
4,647a2198bf8be67e27fb4186,jgw-vanilla-8398dcc3685c4064-UZB -> GBR,jgw-vanilla,UZB,UZB -> GBR,08000 121 700,Seasonal Worker Scheme - Generic,,2023-06-02T17:06:32.998Z,2023-06-02T17:06:32.998Z,0


In [16]:
# generics
def count_unique(df, field):
    return df[field].nunique()

In [17]:
start_date = "2023-05-01" 
end_date = "2023-05-31"
mask = hc_df["createdAt"].between(start_date, end_date, inclusive="both")
hc_df = hc_df.loc[mask]

In [18]:
# phone numbers and emails by route    
def df_aggregate_by_group(df, result_fields, group_field):
    df[group_field] = df[group_field].replace("", "Not specified")
    
    aggregation_dict = {field: lambda values: values.count() for field in result_fields}
    grouped_data = df.groupby(group_field)[result_fields].agg(aggregation_dict)
    
    result = {
        "labels": grouped_data.index.values,
    }
    
    for field in result_fields:
        result[field] = grouped_data[field].values
    
    return result

In [19]:
def dictionary_from_two_lists(labels, values):
    """Creates a dictionary from two lists

    Args:
        labels (_list_): to be used as keys
        values (_list_): to be used as values

    Returns:
        _type_: returns a dictionary 
    """
    new_dict = {}

    for label, value in zip(labels, values):
        new_dict[label] = value

    return new_dict


_df = hc_df
emails_and_phonenumbers_clicks_by_route = df_aggregate_by_group(
    _df,
    result_fields=["phoneNumber", "email"],
    group_field="route"
)
emails_and_phonenumbers_clicks_by_org = df_aggregate_by_group(
    _df,
    result_fields=["phoneNumber", "email"],
    group_field="organization"
)
phone_numbers_by_route = dictionary_from_two_lists(
    emails_and_phonenumbers_clicks_by_route.get("labels"),
    emails_and_phonenumbers_clicks_by_route.get("phoneNumber")
)
emails_by_route = dictionary_from_two_lists(
    emails_and_phonenumbers_clicks_by_route.get("labels"),
    emails_and_phonenumbers_clicks_by_route.get("email")
)

phone_numbers_by_org = dictionary_from_two_lists(
    emails_and_phonenumbers_clicks_by_org.get("labels"),
    emails_and_phonenumbers_clicks_by_org.get("phoneNumber")
)
emails_by_org = dictionary_from_two_lists(
    emails_and_phonenumbers_clicks_by_org.get("labels"),
    emails_and_phonenumbers_clicks_by_org.get("email")
)
print({
    "phone numbers clicks by route": phone_numbers_by_route, 
    "emails clicks by route": emails_by_route,
    # "phone_numbers_by_org":phone_numbers_by_org,
    # "emails_by_org":emails_by_org
})
phone_numbers_by_route, emails_by_route

{'phone numbers clicks by route': {'GBR -> GBR': 1, 'TJK -> GBR': 1}, 'emails clicks by route': {'GBR -> GBR': 1, 'TJK -> GBR': 1}}


({'GBR -> GBR': 1, 'TJK -> GBR': 1}, {'GBR -> GBR': 1, 'TJK -> GBR': 1})